In [1]:
import torch

In [2]:
import gpuclustersim

In [3]:
gpuclustersim.init_device_group("./device_configs/V100.json", 100)

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

device = torch.device("gpuclustersim")

class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(10, 20)
        self.fc2 = nn.Linear(20, 20)
        self.fc3 = nn.Linear(20, 1)
    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        return self.fc3(x)

class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=16, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        self.conv2 = nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, padding=1)
        self.fc = nn.Linear(32 * 7 * 7, 10) 

    def forward(self, x):
        x = self.pool(torch.relu(self.conv1(x)))
        x = self.pool(torch.relu(self.conv2(x)))
        x = torch.flatten(x, 1) 
        return self.fc(x)

class SimpleRNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.rnn = nn.RNN(input_size=10, hidden_size=20, num_layers=1, batch_first=True)
        self.fc = nn.Linear(20, 2)

    def forward(self, x):
        out, hidden = self.rnn(x)
        last_out = out[:, -1, :] 
        return self.fc(last_out)

class SimpleLSTM(nn.Module):
    def __init__(self):
        super().__init__()
        self.lstm = nn.LSTM(input_size=10, hidden_size=20, num_layers=1, batch_first=True)
        self.fc = nn.Linear(20, 2)

    def forward(self, x):
        out, (hn, cn) = self.lstm(x) 
        last_out = out[:, -1, :]
        return self.fc(last_out)

class SimpleTransformer(nn.Module):
    def __init__(self):
        super().__init__()
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=16, 
            nhead=2, 
            dim_feedforward=32, 
            batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=1)
        self.fc = nn.Linear(16, 2)

    def forward(self, x):
        out = self.transformer(x)
        out = out.mean(dim=1) 
        return self.fc(out)

class SimpleResidual(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(10, 10)
        self.fc2 = nn.Linear(10, 10)

    def forward(self, x):
        identity = x
        out = torch.relu(self.fc1(x))
        out = self.fc2(out)
        out += identity # Explicit element-wise addition
        return torch.relu(out)

In [5]:
model = MLP().to(device)
optimizer = optim.SGD(model.parameters(), lr=0.01)
criterion = nn.MSELoss()

In [6]:
for step in range(1):
  x = torch.randn(32, 10).to(device)
  y = torch.randn(32, 1).to(device)
  optimizer.zero_grad()
  loss = criterion(model(x), y)
  loss.backward()
  optimizer.step()

In [7]:
gpuclustersim.save_timeline("./runs/mlp.json")
gpuclustersim.reset_timeline()